# Project RIPRA (ऋप्र): Shack-Hartmann Wavefront Reconstruction & Turbulence Characterization
This notebook runs the data generation, model definition, and training pipeline in Kaggle with GPU acceleration.

### Objectives:
1. Generate a massive synthetic Kolmogorov dataset based on physical Shack-Hartmann parameters.
2. Train a Fully Connected Network (MLP) as a baseline.
3. Train a Spatial 2D ResNet CNN mapping displacements to Zernike coefficients.

## Step 1: Write System Configuration & Calibration Spot Coordinates
We write the system configuration and the 127 active sub-aperture spot coordinates (calibrated from flat frame) to local files.

In [ ]:
import os

# Create directories
os.makedirs('config', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('data_ai', exist_ok=True)
os.makedirs('ml_checkpoints', exist_ok=True)

# Write configuration
system_conf = """# RIPPA system configuration
# All values with SI units unless noted.
# Lines starting with # are comments. Unknown keys are ignored.

# --- Camera ---
camera_pixsize = 7.4e-6       # pixel size (m)
frame_width    = 648           # detector width (pixels)
frame_height   = 492           # detector height (pixels)

# --- Microlens Array (MLA) ---
totlenses      = 127          # number of lenslets
flength        = 18e-3        # focal length (m)
pitch          = 300e-6       # lenslet pitch (m)
sa_radius      = 150e-6       # sub-aperture radius = pitch/2 (m)

# --- Pupil ---
pupil_radius   = 2e-3         # pupil radius (m), diameter = 4 mm

# --- Wavelength ---
wavelength     = 632.8e-9     # HeNe laser (m)

# --- Centroiding ---
thresh_binary  = 0.08         # binary image threshold (relative)
centroid_percent = 0.2         # relative threshold for CoG
coarse_grid_radius = 12        # pixels, coarse grid half-window

# --- Zernike modal reconstruction ---
zernike_nmax   = 5             # max radial order (Noll indexing)

# --- Deformable Mirror ---
dm_nact_x      = 12            # actuators across X
dm_nact_y      = 12            # actuators across Y
coupling       = 0.15          # inter-actuator coupling coefficient
"""
with open('config/system.conf', 'w') as f:
    f.write(system_conf)

# Write spot coordinates
spots_csv = """Spot_ID,ref_cx,ref_cy,col_min,col_max,row_min,row_max
0,336.2358,35.6055,322,350,21,49
1,295.2755,35.9534,281,309,22,50
2,377.0329,35.6968,363,391,22,50
3,254.5339,36.5392,240,268,22,50
4,417.2349,35.4365,403,431,21,49
5,458.3039,35.1009,444,472,21,49
6,214.3198,36.2977,200,228,22,50
7,315.9032,71.2147,302,330,56,84
8,356.8719,71.0683,343,371,56,84
9,397.6367,71.0091,383,411,56,84
10,437.7386,70.9055,423,451,56,84
11,478.4937,70.6403,464,492,56,84
12,193.5659,71.5324,179,207,57,85
13,275.1602,71.1830,261,289,57,85
14,234.8195,71.8606,220,248,57,85
15,377.6086,106.4019,363,391,92,120
16,417.6390,106.0511,403,431,91,119
17,499.1909,105.7953,485,513,91,119
18,173.3499,106.5880,159,187,92,120
19,214.6182,106.3029,200,228,92,120
20,254.8118,106.5430,240,268,92,120
21,295.3025,106.9074,281,309,93,121
22,336.6785,106.8602,323,351,93,121
23,458.8357,106.0552,445,473,92,120
24,397.6551,141.2530,383,411,127,155
25,519.5505,140.5771,505,533,126,154
26,315.6752,142.3981,301,329,128,156
27,438.1966,141.2933,424,452,127,155
28,478.6806,141.1184,464,492,127,155
29,194.1840,142.1445,180,208,127,155
30,235.1020,141.8624,221,249,128,156
31,356.8226,141.8325,342,370,128,156
32,153.1564,141.7089,139,167,127,155
33,275.4212,142.1891,261,289,128,156
34,417.7689,176.4891,403,431,162,190
35,376.8731,176.8554,363,391,162,190
36,255.1783,177.3068,240,268,163,191
37,295.8705,177.2695,281,309,163,191
38,336.3475,177.1932,322,350,162,190
39,458.3711,176.3204,444,472,162,190
40,499.0494,176.0652,485,513,161,189
41,540.0303,176.1309,525,553,162,190
42,174.1007,177.1961,160,188,163,191
43,214.6520,177.1700,200,228,163,191
44,132.8550,177.0232,118,146,162,190
45,194.6446,212.2194,180,208,197,225
46,356.8366,212.0724,342,370,198,226
47,397.4077,212.0187,383,411,197,225
48,560.2516,211.2724,546,574,197,225
49,235.6775,212.2453,221,249,197,225
50,275.9207,212.2514,261,289,198,226
51,316.0615,212.0098,302,330,197,225
52,437.9741,211.9568,423,451,198,226
53,479.0589,211.4635,465,493,197,225
54,519.2873,211.4520,505,533,197,225
55,112.7143,212.1075,98,126,197,225
56,153.8417,212.7019,139,167,198,226
57,417.9044,246.8852,404,432,232,260
58,255.9168,247.1785,241,269,232,260
59,336.6092,247.3355,322,350,233,261
60,458.6012,246.9010,443,471,232,260
61,580.6331,247.0862,566,594,233,261
62,215.2415,247.3441,200,228,233,261
63,295.8578,247.4524,281,309,233,261
64,377.0680,247.1545,362,390,233,261
65,499.1489,247.0840,484,512,233,261
66,539.6986,246.9290,525,553,232,260
67,133.7119,247.6281,119,147,233,261
68,174.1644,247.4453,160,188,233,261
69,92.9833,248.4109,79,107,234,262
70,356.9891,281.9113,343,371,267,295
71,397.8587,282.2310,383,411,268,296
72,438.4990,282.2587,424,452,268,296
73,194.8801,282.6187,181,209,268,296
74,316.1070,282.1281,301,329,267,295
75,479.1132,282.0064,464,492,268,296
76,519.7640,281.8660,505,533,267,295
77,560.8694,281.8977,546,574,267,295
78,153.9710,282.8376,139,167,268,296
79,235.9145,282.6279,221,249,268,296
80,275.8482,282.7121,260,288,268,296
81,112.7811,283.5667,99,127,269,297
82,336.8174,317.1207,322,350,302,330
83,296.2087,317.3956,281,309,303,331
84,377.8269,317.1841,363,391,303,331
85,418.6037,317.8509,404,432,303,331
86,499.4087,317.0190,485,513,303,331
87,540.2759,317.1366,525,553,302,330
88,174.3854,318.0666,160,188,303,331
89,215.2590,317.8863,201,229,303,331
90,255.9398,317.6587,241,269,303,331
91,458.8821,317.5141,444,472,303,331
92,133.5800,318.0774,119,147,304,332
93,479.2986,352.4147,465,493,338,366
94,357.6485,353.0772,343,371,339,367
95,438.6258,352.7084,424,452,338,366
96,235.3956,353.0910,220,248,339,367
97,276.1352,353.0891,262,290,339,367
98,316.4901,353.0632,301,329,339,367
99,398.1995,353.0821,384,412,339,367
100,520.0254,352.5521,505,533,338,366
101,153.7353,353.5481,139,167,339,367
102,194.8299,353.2079,180,208,339,367
103,378.2270,388.0141,363,391,373,401
104,337.2311,388.0485,323,351,373,401
105,459.0131,387.5397,445,473,373,401
106,296.6242,388.0533,282,310,373,401
107,418.5550,387.9681,404,432,373,401
108,174.9062,387.6528,160,188,373,401
109,255.7890,388.0385,241,269,374,402
110,499.7140,387.7201,485,513,373,401
111,215.2666,388.2955,201,229,374,402
112,317.3679,423.2000,303,331,409,437
113,357.8784,423.0118,343,371,409,437
114,398.3787,422.8690,384,412,408,436
115,438.8049,422.8808,424,452,409,437
116,479.6269,422.7636,465,493,408,436
117,236.0765,423.3589,222,250,409,437
118,276.1863,423.2817,261,289,409,437
119,194.9470,424.0562,180,208,410,438
120,378.1558,458.9573,363,391,445,473
121,418.7422,458.4331,404,432,444,472
122,256.3637,458.9417,242,270,444,472
123,296.7686,459.0362,282,310,444,472
124,337.5879,458.6027,323,351,445,473
125,459.3836,458.3389,444,472,444,472
126,215.2769,459.2897,201,229,445,473
"""
with open('results/reference_centroids_c.csv', 'w') as f:
    f.write(spots_csv)

print('Successfully created configuration and reference spots CSV files!')

## Step 2: Define Dataset Generator
This class generates synthetic Kolmogorov turbulence frames with physical scaling and adds readout noise.

In [ ]:
#!/usr/bin/env python3
"""
generate_dataset.py - Generate synthetic Kolmogorov turbulence dataset for Shack-Hartmann WFS
"""
import os
import argparse
import numpy as np
import scipy.special as special
import pandas as pd

def noll_to_nm(j):
    """Convert Noll index j (1-based) to radial order n and azimuthal frequency m"""
    if j == 1:
        return 0, 0
    current_j = 2
    for n in range(1, 100):
        for m in range(n % 2, n + 1, 2):
            if m == 0:
                if current_j == j:
                    return n, 0
                current_j += 1
            else:
                # Same parity
                if current_j % 2 == 1:
                    if current_j == j:
                        return n, -m
                    if current_j + 1 == j:
                        return n, m
                else:
                    if current_j == j:
                        return n, m
                    if current_j + 1 == j:
                        return n, -m
                current_j += 2

def noll_covariance(n1, m1, j1, n2, m2, j2):
    """Compute the Noll covariance coefficient C_ij for Kolmogorov turbulence"""
    if abs(m1) != abs(m2):
        return 0.0
    # Parity check: both must be even or both odd Noll indices for same m
    if (j1 % 2) != (j2 % 2) and abs(m1) != 0:
        return 0.0
    
    # Sign factor
    sign = (-1.0) ** ((n1 + n2 - 2 * abs(m1)) / 2)
    
    # Gamma terms
    num = (sign * np.sqrt((n1 + 1) * (n2 + 1)) * 
           special.gamma(14.0 / 3.0) * 
           special.gamma((n1 + n2 - 5.0 / 3.0) / 2.0))
    den = (special.gamma((n1 - n2 + 17.0 / 3.0) / 2.0) * 
           special.gamma((n2 - n1 + 17.0 / 3.0) / 2.0) * 
           special.gamma((n1 + n2 + 23.0 / 3.0) / 2.0))
    
    return num / den

def get_noll_covariance_matrix(max_j):
    """Compute the Noll covariance matrix for j = 2 to max_j (excluding piston)"""
    nmodes = max_j - 1
    C = np.zeros((nmodes, nmodes))
    modes = []
    for idx in range(nmodes):
        j = idx + 2
        n, m = noll_to_nm(j)
        modes.append((j, n, m))
        
    for i in range(nmodes):
        j1, n1, m1 = modes[i]
        for j_idx in range(nmodes):
            j2, n2, m2 = modes[j_idx]
            C[i, j_idx] = noll_covariance(n1, m1, j1, n2, m2, j2)
            
    return C, modes

# Zernike radial coefficient Helper
def zernike_coeff(n, m, s):
    abs_m = abs(m)
    num = special.factorial(n - s)
    den = special.factorial(s) * special.factorial((n + abs_m)//2 - s) * special.factorial((n - abs_m)//2 - s)
    sign = 1.0 if s % 2 == 0 else -1.0
    return sign * num / den

# Evaluate Zernike derivatives at normalized polar/cartesian coordinate
def zernike_derivatives(n, m, x, y):
    rho = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)
    abs_m = abs(m)
    
    R = 0.0
    dR = 0.0
    for s in range((n - abs_m) // 2 + 1):
        c = zernike_coeff(n, m, s)
        pow_rho = n - 2*s
        if pow_rho > 0:
            R += c * (rho ** pow_rho)
            dR += c * pow_rho * (rho ** (pow_rho - 1))
        elif pow_rho == 0:
            R += c
            
    norm_factor = np.sqrt(n + 1) if m == 0 else np.sqrt(2 * (n + 1))
    R *= norm_factor
    dR *= norm_factor
    
    cos_mt = np.cos(abs_m * theta)
    sin_mt = np.sin(abs_m * theta)
    
    if m >= 0:
        dz_drho = dR * cos_mt
        dz_dtheta = -abs_m * R * sin_mt
    else:
        dz_drho = dR * sin_mt
        dz_dtheta = abs_m * R * cos_mt
        
    if rho < 1e-9:
        if n == 1:
            if m == 1:
                return norm_factor, 0.0
            elif m == -1:
                return 0.0, norm_factor
        return 0.0, 0.0
        
    dzdx = dz_drho * np.cos(theta) - dz_dtheta * np.sin(theta) / rho
    dzdy = dz_drho * np.sin(theta) + dz_dtheta * np.cos(theta) / rho
    return dzdx, dzdy

def load_system_config(config_path):
    cfg = {}
    with open(config_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if '=' in line:
                key, val = line.split('=', 1)
                key = key.strip()
                val = val.split('#', 1)[0].strip()
                try:
                    cfg[key] = float(val) if '.' in val or 'e' in val else int(val)
                except ValueError:
                    cfg[key] = val
    return cfg

def main():
    parser = argparse.ArgumentParser(description="Generate synthetic Shack-Hartmann dataset")
    parser.add_argument("--samples", type=int, default=10000, help="Number of samples to generate")
    parser.add_argument("--seed", type=int, default=42, help="Random seed")
    parser.add_argument("--noise", type=float, default=0.1, help="Standard deviation of displacement noise in pixels")
    parser.add_argument("--out", type=str, default="data_ai/dataset.npz", help="Output filepath")
    args = parser.parse_args()
    
    np.random.seed(args.seed)
    
    # 1. Load config and spot coordinates
    print("Loading system configuration...")
    cfg = load_system_config("config/system.conf")
    
    spots_file = "results/reference_centroids_c.csv"
    if not os.path.exists(spots_file):
        print(f"Error: {spots_file} not found. Please run test_centroid first to calibrate.")
        return
        
    spots_df = pd.read_csv(spots_file)
    nspots = len(spots_df)
    print(f"Loaded {nspots} active sub-aperture coordinates.")
    
    # 2. Setup Zernike covariance
    zernike_nmax = int(cfg["zernike_nmax"])
    max_j = (zernike_nmax + 1) * (zernike_nmax + 2) // 2
    nmodes = max_j - 1 # exclude piston
    print(f"Setting up Zernike covariance for {nmodes} modes (Noll radial order <= {zernike_nmax})...")
    
    C, modes = get_noll_covariance_matrix(max_j)
    
    # 3. Precompute Zernike Derivative Matrix (Zprime) via area integration
    print("Precomputing Zernike derivative matrix (Zprime)...")
    M = 15 # 15x15 quadrature grid
    rbar = cfg["sa_radius"] / cfg["pupil_radius"]
    kk = cfg["pupil_radius"] / (np.pi * cfg["sa_radius"]**2)
    
    Zprime = np.zeros((2 * nspots, nmodes))
    
    for k in range(nspots):
        ref_cx = spots_df.loc[k, "ref_cx"]
        ref_cy = spots_df.loc[k, "ref_cy"]
        
        # Canonical center coordinates (origin at pupil center, Y pointing up)
        x_c = (ref_cx - spots_df["ref_cx"].mean()) * cfg["camera_pixsize"] / cfg["pupil_radius"]
        y_c = -(ref_cy - spots_df["ref_cy"].mean()) * cfg["camera_pixsize"] / cfg["pupil_radius"]
        
        for m_idx, (j, n, m) in enumerate(modes):
            sum_dzdx = 0.0
            sum_dzdy = 0.0
            count_pts = 0
            
            # 2D area integration over sub-aperture disk
            for r_step in range(M):
                dy = -rbar + 2.0 * rbar * r_step / (M - 1)
                for c_step in range(M):
                    dx = -rbar + 2.0 * rbar * c_step / (M - 1)
                    if dx**2 + dy**2 <= rbar**2:
                        dzdx, dzdy = zernike_derivatives(n, m, x_c + dx, y_c + dy)
                        sum_dzdx += dzdx
                        sum_dzdy += dzdy
                        count_pts += 1
                        
            avg_dzdx = sum_dzdx / count_pts if count_pts > 0 else 0.0;
            avg_dzdy = sum_dzdy / count_pts if count_pts > 0 else 0.0;
            
            # Map to Zprime elements
            Zprime[k, m_idx] = kk * avg_dzdx
            Zprime[k + nspots, m_idx] = -kk * avg_dzdy
            
    # 4. Generate Samples using AR(1) Temporal Correlation
    print(f"Generating {args.samples} samples (arranged in temporally correlated sequences)...")
    displacements = np.zeros((args.samples, 2 * nspots))
    coefficients = np.zeros((args.samples, nmodes))
    D_r0_arr = np.zeros(args.samples)
    
    # Scale factor mapping coefficients in meters to displacements in pixels
    scale_factor = cfg["flength"] / cfg["camera_pixsize"]
    
    # Define sequence lengths (default 1000 frames per sequence representing 1 second at 1000 Hz)
    seq_len = 1000
    n_seq = args.samples // seq_len
    if n_seq == 0:
        n_seq = 1
        seq_len = args.samples
        
    print(f"  Configuration: {n_seq} sequences of length {seq_len} frames.")
    
    frame_idx = 0
    for s in range(n_seq):
        # Turbulence strength for this sequence
        D_r0 = np.random.uniform(1.0, 10.0)
        # Coherence time for this sequence: tau0 in range 2 ms to 10 ms
        tau0 = np.random.uniform(0.002, 0.010)
        fs = 1000.0 # 1000 Hz frame rate (1 ms interval)
        
        # AR(1) coefficient: rho = exp(-dt / tau0)
        rho = np.exp(-1.0 / (fs * tau0))
        
        # Scale covariance by (D/r0)^(5/3)
        cov = C * (D_r0 ** (5.0 / 3.0))
        reg_cov = cov + np.eye(nmodes) * 1e-8
        
        # Draw starting coefficients for the sequence
        a_rad = np.random.multivariate_normal(np.zeros(nmodes), reg_cov)
        
        for t in range(seq_len):
            if t > 0:
                # AR(1) step: update Zernike coefficients with temporal correlation
                epsilon = np.random.multivariate_normal(np.zeros(nmodes), reg_cov)
                a_rad = rho * a_rad + np.sqrt(1.0 - rho * rho) * epsilon
                
            # Convert to meters for physical slope mapping: a_m = a_rad * (lambda / 2pi)
            a_m = a_rad * (cfg["wavelength"] / (2.0 * np.pi))
            
            # Calculate clean displacements in pixels
            clean_disp = Zprime.dot(a_m) * scale_factor
            
            # Add noise
            noise = np.random.normal(0.0, args.noise, 2 * nspots)
            noisy_disp = clean_disp + noise
            
            displacements[frame_idx] = noisy_disp
            coefficients[frame_idx] = a_rad
            D_r0_arr[frame_idx] = D_r0
            
            frame_idx += 1
            if frame_idx >= args.samples:
                break
                
        print(f"  Sequence {s + 1:02d}/{n_seq:02d} complete (D/r0={D_r0:.2f}, tau0={tau0*1000.0:.2f} ms).")
        
    # 5. Save dataset
    out_dir = os.path.dirname(args.out)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        
    np.savez(args.out, 
             displacements=displacements, 
             coefficients=coefficients,
             D_r0=D_r0_arr)
             
    print(f"Successfully saved dataset to {args.out}!")
    print(f"  Inputs shape:  {displacements.shape}")
    print(f"  Targets shape: {coefficients.shape}")

if __name__ == "__main__":
    main()


## Step 3: Define PyTorch Models (MLP and ResNet-CNN)

In [ ]:
# ml/models.py - PyTorch Neural Network Architectures for Wavefront Reconstruction
import torch
import torch.nn as nn

class WavefrontMLP(nn.Module):
    """
    Fully Connected Multi-Layer Perceptron Baseline model.
    Maps 1D vector of displacements directly to Zernike coefficients.
    """
    def __init__(self, input_dim=254, output_dim=20, hidden_dims=[512, 256, 128], dropout=0.1):
        super(WavefrontMLP, self).__init__()
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.LayerNorm(h_dim))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x)


class ResBlock(nn.Module):
    """
    Standard Residual block with BatchNorm and Relu activation.
    """
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.relu(out)
        return out


class WavefrontCNN(nn.Module):
    """
    Spatial 2D Convolutional Neural Network Reconstructor.
    Maps 2-channel 2D grid of spot displacements (dx, dy) to Zernike coefficients.
    """
    def __init__(self, output_dim=20):
        super(WavefrontCNN, self).__init__()
        self.conv1 = nn.Conv2d(2, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()
        
        self.res1 = ResBlock(32)
        self.res2 = ResBlock(32)
        
        # Downsample using strided convolution (13x13 -> 7x7)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.res3 = ResBlock(64)
        
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        self.fc = nn.Sequential(
            nn.Linear(64 * 3 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, output_dim)
        )
        
    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.res1(out)
        out = self.res2(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.res3(out)
        out = self.pool(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)


## Step 4: Define Training & Validation Pipeline

In [ ]:
# ml/train.py - Train PyTorch models for Shack-Hartmann wavefront reconstruction
import os
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split



class WavefrontDataset(Dataset):
    """
    Dataset loader mapping 1D displacements to Zernike coefficients.
    If mode is 'cnn', handles spatial mapping of the 127 spots to a 2D grid.
    """
    def __init__(self, dataset_path, spots_csv_path, model_type='mlp'):
        self.model_type = model_type
        
        # Load npz dataset
        data = np.load(dataset_path)
        self.displacements = torch.tensor(data['displacements'], dtype=torch.float32)
        self.coefficients = torch.tensor(data['coefficients'], dtype=torch.float32)
        self.n_samples = len(self.displacements)
        self.nspots = self.displacements.shape[1] // 2
        
        # Load spot coordinates for spatial CNN mapping
        spots_df = pd.read_csv(spots_csv_path)
        
        # Estimate pitch and center
        mean_cx = spots_df["ref_cx"].mean()
        mean_cy = spots_df["ref_cy"].mean()
        
        # Approximate pitch in pixels (~40.1 px)
        # Find minimum distance to neighbor for pitch
        dists = []
        for i in range(len(spots_df)):
            dx = spots_df["ref_cx"].values - spots_df["ref_cx"].values[i]
            dy = spots_df["ref_cy"].values - spots_df["ref_cy"].values[i]
            d = np.hypot(dx, dy)
            d = d[d > 1e-3]
            if len(d) > 0:
                dists.append(d.min())
        pitch_px = np.mean(dists) if len(dists) > 0 else 40.1
        
        # Map spots to integer grid indices (u, v)
        u_coords = np.round((spots_df["ref_cx"].values - mean_cx) / pitch_px).astype(int)
        v_coords = np.round((spots_df["ref_cy"].values - mean_cy) / pitch_px).astype(int)
        
        u_min, u_max = u_coords.min(), u_coords.max()
        v_min, v_max = v_coords.min(), v_coords.max()
        
        self.grid_w = int(u_max - u_min + 1)
        self.grid_h = int(v_max - v_min + 1)
        self.u_offset = int(-u_min)
        self.v_offset = int(-v_min)
        
        self.spot_grid_coords = list(zip(u_coords, v_coords))
        
    def __len__(self):
        return self.n_samples
        
    def __getitem__(self, idx):
        disp = self.displacements[idx]
        coeff = self.coefficients[idx]
        
        if self.model_type == 'mlp':
            return disp, coeff
        else: # cnn
            # Arrange 1D displacements into 2-channel 2D spatial grid (dx, dy)
            grid = torch.zeros((2, self.grid_h, self.grid_w), dtype=torch.float32)
            for k in range(self.nspots):
                u, v = self.spot_grid_coords[k]
                row = v + self.v_offset
                col = u + self.u_offset
                grid[0, row, col] = disp[k]
                grid[1, row, col] = disp[k + self.nspots]
            return grid, coeff

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(dataloader.dataset)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * inputs.size(0)
    return running_loss / len(dataloader.dataset)

def main():
    parser = argparse.ArgumentParser(description="Train Shack-Hartmann neural network reconstructor")
    parser.add_argument("--model", type=str, default="mlp", choices=["mlp", "cnn"], help="Model type to train")
    parser.add_argument("--epochs", type=int, default=50, help="Number of epochs to train")
    parser.add_argument("--batch_size", type=int, default=64, help="Batch size")
    parser.add_argument("--lr", type=float, default=1e-3, help="Learning rate")
    parser.add_argument("--dataset", type=str, default="data_ai/dataset.npz", help="Dataset path")
    parser.add_argument("--out_dir", type=str, default="ml_checkpoints", help="Output checkpoints directory")
    args = parser.parse_args()
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using execution device: {device}")
    
    # 1. Load dataset
    spots_csv = "results/reference_centroids_c.csv"
    if not os.path.exists(spots_csv):
        print(f"Error: {spots_csv} not found. Please calibrate first using C pipeline.")
        return
        
    print(f"Loading dataset from {args.dataset}...")
    full_dataset = WavefrontDataset(args.dataset, spots_csv, model_type=args.model)
    
    # Train / Val / Test split (80% / 10% / 10%)
    total_len = len(full_dataset)
    train_len = int(0.8 * total_len)
    val_len = int(0.1 * total_len)
    test_len = total_len - train_len - val_len
    
    train_set, val_set, test_set = random_split(
        full_dataset, [train_len, val_len, test_len], 
        generator=torch.Generator().manual_seed(42)
    )
    
    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=args.batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=args.batch_size, shuffle=False)
    
    print(f"Dataset split: Train={train_len}, Val={val_len}, Test={test_len}")
    
    # 2. Setup model
    if args.model == "mlp":
        model = WavefrontMLP(input_dim=full_dataset.nspots * 2, output_dim=full_dataset.coefficients.shape[1])
    else: # cnn
        model = WavefrontCNN(output_dim=full_dataset.coefficients.shape[1])
        
    model = model.to(device)
    print(model)
    
    # Loss & Optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    
    # 3. Training Loop
    os.makedirs(args.out_dir, exist_ok=True)
    best_val_loss = float("inf")
    
    print("\nStarting Training...")
    for epoch in range(args.epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        
        print(f"Epoch {epoch+1:02d}/{args.epochs:02d} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")
        
        # Save best checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            checkpoint_path = os.path.join(args.out_dir, f"best_{args.model}.pt")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, checkpoint_path)
            
    # 4. Final Evaluation
    checkpoint_path = os.path.join(args.out_dir, f"best_{args.model}.pt")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    test_loss = evaluate(model, test_loader, criterion, device)
    print(f"\nFinal Test MSE Loss on Best Checkpoint: {test_loss:.6f}")
    
    # Compare with C Reconstructor on 5 test samples
    model.eval()
    print("\nVisual spot check: True vs Predicted Zernike coeffs (first 5 modes of sample 1):")
    with torch.no_grad():
        inputs, targets = next(iter(test_loader))
        inputs, targets = inputs.to(device), targets.to(device)
        preds = model(inputs)
        for i in range(5):
            print(f"  Sample {i+1}: True={[float(x) for x in targets[i][:5]]} | Pred={[float(x) for x in preds[i][:5]]}")

if __name__ == "__main__":
    main()


## Step 5: Generate Massive Dataset
Generate 50,000 samples of synthetic Kolmogorov turbulence.

In [ ]:
# We will run the script as a python subprocess or import and run main
# Run generate_dataset with custom args
import sys
sys.argv = ['generate_dataset.py', '--samples', '50000', '--out', 'data_ai/dataset.npz', '--noise', '0.1']
main() # Runs the main from generate_dataset.py

## Step 6: Train WavefrontMLP Reconstructor
Train the MLP baseline for 30 epochs.

In [ ]:
# We overwrite the args and call main from train.py
import sys
sys.argv = ['train.py', '--model', 'mlp', '--epochs', '30', '--batch_size', '128', '--lr', '1e-3']
# In train.py, the main is defined at the module scope
import train
train.main()

## Step 7: Train WavefrontCNN Reconstructor
Train the ResNet-based CNN reconstructor for 30 epochs.

In [ ]:
import sys
sys.argv = ['train.py', '--model', 'cnn', '--epochs', '30', '--batch_size', '128', '--lr', '1e-3']
import train
train.main()